# Submissão 2B — LSTM (FastText, sem Bidirectional)

In [1]:
import sys
import pandas as pd
import torch
import torch.nn as nn

sys.path.append('..')
from src.data_utils import encode_texts
from src.pytorch_models import select_device

device = select_device()
print('Device:', device)

Device: cpu


In [6]:
artifact = torch.load('lstm_fasttext_noBI.pt', map_location=device)

vocab        = artifact['vocab']
max_len      = artifact['max_len']
class_order  = artifact['class_order']
label_to_idx = artifact['label_to_idx']
idx_to_label = {v: k for k, v in label_to_idx.items()}
cfg          = artifact['config']

print('Classes:', class_order)
print('Config:', cfg)

Classes: ['Human', 'Google', 'Meta', 'OpenAI', 'Anthropic']
Config: {'embed_dim': 300, 'hidden_dim': 128, 'num_layers': 1, 'dropout': 0.3}


In [7]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, num_classes, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        return self.classifier(hidden[-1])


model = LSTMClassifier(
    vocab_size=len(vocab),
    embed_dim=cfg['embed_dim'],
    hidden_dim=cfg['hidden_dim'],
    num_layers=cfg['num_layers'],
    num_classes=len(class_order),
    dropout=cfg['dropout'],
).to(device)

model.load_state_dict(artifact['state_dict'])
model.eval()
print('Modelo carregado.')

Modelo carregado.


In [8]:
df_subm = pd.read_csv('../data/subm2.csv', sep=';')
print(f'Textos a classificar: {len(df_subm)}')
df_subm.head()

Textos a classificar: 150


,ID,Text
0,D2-101,Microbial mats of coexisting bacteria and arch...
1,D2-102,The origin of life on Earth remains a complex ...
2,D2-103,Estimates of the time at which life arose on E...
3,D2-104,Life on Earth emerged roughly 3.8-4 billion ye...
4,D2-105,Black holes predominantly form from the catast...


In [9]:
import numpy as np

X = encode_texts(df_subm['Text'].values, vocab, max_len)
x_tensor = torch.tensor(X, dtype=torch.long, device=device)

with torch.no_grad():
    logits = model(x_tensor)
    pred_idx = logits.argmax(dim=1).cpu().tolist()

pred_labels = [idx_to_label[i] for i in pred_idx]

df_out = pd.DataFrame({'ID': df_subm['ID'], 'Labels': pred_labels})

print('Label counts:')
print(df_out['Labels'].value_counts())
df_out.head()

Label counts:
Labels
Anthropic    53
Human        51
OpenAI       25
Meta         12
Google        9
Name: count, dtype: int64


,ID,Labels
0,D2-101,Human
1,D2-102,OpenAI
2,D2-103,Human
3,D2-104,Anthropic
4,D2-105,Human


In [10]:
output_path = 'subm2-g1-MEI-B.csv'
df_out.to_csv(output_path, index=False, sep=';')
print('Guardado:', output_path)

Guardado: subm2-g1-MEI-B.csv
